# Hyperview — Hydra Runner
Mount Drive, install dependencies, then run  with any config override.
All experiment config is in  — **do not edit ** to change hyperparameters.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess

# ---------- paths ----------
PROJECT_DIR  = "/content/drive/MyDrive/hyperview"   # project root on Drive
TRAIN_TAR    = "/content/drive/MyDrive/train.tar.gz"
TEST_TAR     = "/content/drive/MyDrive/test.tar.gz"
TRAIN_GT     = "/content/drive/MyDrive/train_gt.csv"
WAVELENGTHS  = "/content/drive/MyDrive/wavelengths.csv"

# ---------- extract data ----------
os.makedirs("/content/train_data", exist_ok=True)
os.makedirs("/content/test_data",  exist_ok=True)
!tar -xzf {TRAIN_TAR} -C /content/train_data
!tar -xzf {TEST_TAR}  -C /content/test_data
!cp {TRAIN_GT}    /content/train_gt.csv
!cp {WAVELENGTHS} /content/wavelengths.csv

# ---------- change working dir to project ----------
os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# Install dependencies (run once per Colab session)
!pip install -q hydra-core omegaconf timm wandb torchgeo joblib tqdm

In [ ]:
# ── W&B API key setup ──────────────────────────────────────────────
# Option A (recommended): store key in Colab Secrets
#   Runtime → Manage secrets → add  WANDB_API_KEY  with your key
import os
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B API key loaded from Colab Secrets.")
except Exception:
    # Option B: paste key directly (never commit this to git)
    # os.environ["WANDB_API_KEY"] = "paste-your-key-here"
    print("WANDB_API_KEY not found in Colab Secrets."
          " wandb.login() will prompt interactively, or set it above.")

# Option C: pass key at runtime (no env var needed):
#   python train.py wandb.api_key=your_key_here


## Run training
Use Hydra CLI overrides to switch experiments **without editing any file**.

| Config key | Purpose |
|---|---|
| `model=convnext_tiny` | switch backbone (PCA mode) |
| `model=convnext_small_conv` + `data.spectral_mode=conv` | use conv spectral reducer |
| `data.pca_components=10` | change PCA components (PCA mode) |
| `model.reducer_channels=10` | change reducer output channels (conv mode) |
| `training.loss=smooth_l1` | switch loss function |
| `wandb.enabled=false` | disable W&B |


In [ ]:
# ── Edit the override string to switch experiments ──────────────────────
# PCA mode examples (default):
#   model=convnext_tiny  data.pca_components=10
#   model=convnext_large data.pca_components=5  training.loss=smooth_l1
#   model=efficientnet_b0
#
# Conv reducer mode examples:
#   model=convnext_small_conv data.spectral_mode=conv
#   model=convnext_large_conv data.spectral_mode=conv model.reducer_channels=10
#
# Available PCA  models: convnext_tiny, convnext_small, convnext_large, efficientnet_b0
# Available conv models: convnext_tiny_conv, convnext_small_conv, convnext_large_conv

overrides = "model=convnext_small data.pca_components=3"

!python train.py {overrides}


## Submission inference
Run after training finishes.

In [ ]:
# ── Submission inference ─────────────────────────────────────────────────
# Reads artefacts saved by train.py and produces submission.csv.
# Set SPECTRAL_MODE and BACKBONE_NAME to match the training run.

import os, torch, joblib
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from src.dataset import load_data, transform_patches_with_mask, NPZDataset
from src.model   import HyperspectralRegressor

OUTPUT_DIR    = "/content/outputs"
SPECTRAL_MODE = "pca"                  # "pca" or "conv" — must match training
BACKBONE_NAME = "convnext_small_in22k" # must match training

# ── Load artefacts ───────────────────────────────────────────────────────
scaler_y     = joblib.load(f"{OUTPUT_DIR}/scaler_labels.pkl")
stats        = torch.load(f"{OUTPUT_DIR}/global_stats.pt")
global_means = stats["means"]
global_stds  = stats["stds"]

X_sub = load_data("/content/test_data/test")

if SPECTRAL_MODE == "pca":
    scaler_x     = joblib.load(f"{OUTPUT_DIR}/scaler_hyperspectral.pkl")
    pca          = joblib.load(f"{OUTPUT_DIR}/pca_hyperspectral.pkl")
    X_sub_ds     = transform_patches_with_mask(X_sub, scaler_x, pca)
    in_channels  = pca.n_components_
    use_reducer  = False
    reducer_ch   = 32
else:  # conv
    X_sub_ds     = X_sub
    in_channels  = 3   # must equal model.reducer_channels used during training
    use_reducer  = True
    reducer_ch   = 32  # must equal model.reducer_mid_channels used during training

# ── Dataset / loader ─────────────────────────────────────────────────────
sub_ds     = NPZDataset(X_sub_ds, augment=False,
                        global_means=global_means, global_stds=global_stds)
sub_loader = DataLoader(sub_ds, batch_size=64, shuffle=False,
                        num_workers=2, pin_memory=True)

# ── Model ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = HyperspectralRegressor(
    in_channels          = in_channels,
    backbone_name        = BACKBONE_NAME,
    pretrained           = False,
    use_spectral_reducer = use_reducer,
    reducer_mid_channels = reducer_ch,
).to(device)
model.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/best_model.pth", map_location=device)
)
model.eval()

# ── Inference ────────────────────────────────────────────────────────────
preds_scaled = []
with torch.no_grad():
    for X_b, nv_b in sub_loader:
        preds_scaled.append(
            model(X_b.to(device), nv_b.to(device)).cpu().numpy()
        )

y_pred = scaler_y.inverse_transform(np.vstack(preds_scaled))
pd.DataFrame(y_pred, columns=["P","K","Mg","pH"]).to_csv(
    "submission.csv", index_label="sample_index"
)
print("submission.csv saved")
